In [1]:
import numpy as np
import netket as nk
from math import comb
from scipy.sparse.linalg import eigsh
from netket.experimental.operator import ParticleNumberAndSpinConservingFermioperator2nd

# 为了代码简洁，直接重命名算符，与你的成功案例保持一致
from netket.operator.fermion import create as cdag
from netket.operator.fermion import destroy as c
from netket.operator.fermion import number as nc

# --- 1. 系统参数配置 ---
L = 8                 # 一维链长度
t = 1.0               # 跳跃能级
U = 8.0               # 原位排斥能 (Hubbard U)
n_orbitals = L

# 设置半满配置
if n_orbitals % 2 != 0:
    raise ValueError("n_orbitals 必须为偶数，才能设置 N_up = N_dn = n_orbitals/2")

N_up = n_orbitals // 2 - 1
N_dn = n_orbitals // 2 - 1

# --- 2. 构造【受限的】费米子希尔伯特空间 ---
# 【核心修复】：使用 n_fermions_per_spin 来分别固定上下自旋的电子数
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals, s=1/2, n_fermions_per_spin=(N_up, N_dn)
)

print('n_orbitals =', n_orbitals)
print('fixed particles (N_up, N_dn) =', (N_up, N_dn))

# --- 3. 构造哈密顿量 ---
H = 0.0  

# 动能项 (Hopping) - OBC 开边界
# sum_{i, sigma} -t * (c^dag_{i, sigma} c_{i+1, sigma} + h.c.)
for i in range(L - 1):
    for sz in (+1, -1):
        H += -t * (cdag(hi, i, sz=sz) @ c(hi, i+1, sz=sz))
        H += -t * (cdag(hi, i+1, sz=sz) @ c(hi, i, sz=sz))

# 相互作用项 (Interaction)
# sum_{i} U * n_{i, up} * n_{i, down}
for i in range(L):
    n_up_i = nc(hi, i, sz=+1)
    n_dn_i = nc(hi, i, sz=-1)
    H += U * (n_up_i @ n_dn_i)

# --- 4. 维度评估与守恒量扇区构建 ---
sector_dim = comb(n_orbitals, N_up) * comb(n_orbitals, N_dn)
max_dim_for_sparse_ed = 2000000  

print("\n--- 系统信息 ---")
print(f"L={L}, U={U}, 电子配置=(N_up={N_up}, N_dn={N_dn})")
print(f"预计该扇区希尔伯特空间维度 (Sector Dim) = {sector_dim}")

if sector_dim > max_dim_for_sparse_ed:
    print("[Skip ED] 该扇区维度过大，不执行 to_sparse()/eigsh 以避免内存溢出。")
else:
    # --- 5. 提取守恒子空间并严格对角化 ---
    # 直接传入 H，底层会自动读取 hi 中的 n_fermions_per_spin 约束
    H_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(H)
    
    print("\n正在构建稀疏矩阵...")
    sp_h = H_ed.to_sparse()
    
    print("正在使用 Lanczos 算法求解基态...")
    # 求解前 k 个最小特征值 (SA: Smallest Algebraic)
    k_vals = min(4, sp_h.shape[0] - 2)
    eig_vals, eig_vecs = eigsh(sp_h, k=k_vals, which="SA")
    
    # 特征值从小到大排序
    eig_vals = np.sort(np.real(eig_vals))

# --- 5. 计算并输出结果 ---
    print("\n--- 计算结果 ---")
    print(f"当前守恒扇区的矩阵实际维度: {sp_h.shape[0]}")
    
    # 基态能量 E0
    e0 = eig_vals[0]
    # 单位格点平均能量
    e_per_site = e0 / L

    print(f"单位格点平均能量 E0/L = {e_per_site:.16f}")
    print(f"最低 {len(eig_vals)} 个能级 = {eig_vals}")

C:\Users\10783\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: To build H|ψ⟩ use nk.vqs.apply_operator(H, vstate_ψ).

n_orbitals = 8
fixed particles (N_up, N_dn) = (3, 3)

--- 系统信息 ---
L=8, U=8.0, 电子配置=(N_up=3, N_dn=3)
预计该扇区希尔伯特空间维度 (Sector Dim) = 3136

正在构建稀疏矩阵...
正在使用 Lanczos 算法求解基态...

--- 计算结果 ---
当前守恒扇区的矩阵实际维度: 3136
单位格点平均能量 E0/L = -0.6133250844218981
最低 4 个能级 = [-4.90660068 -4.71796157 -4.4952787  -4.40670521]


In [2]:
import numpy as np

def measure_pairing_correlation(hi, psi, i, j, k, l):
    """
    测量配对关联函数 P(ij, kl) = ⟨Ψ| Δ†_ij Δ_kl |Ψ⟩
    其中 Δ_xy = c_{x↑} c_{y↓} - c_{x↓} c_{y↑}
    
    展开后共计 4 项：
    P(ij, kl) = -(c†_{i↑} c†_{j↓} - c†_{i↓} c†_{j↑}) @ (c_{k↑} c_{l↓} - c_{k↓} c_{l↑})
              = - c†_{i↑} c†_{j↓} c_{k↑} c_{l↓}  (项1)
                + c†_{i↑} c†_{j↓} c_{k↓} c_{l↑}  (项2)
                + c†_{i↓} c†_{j↑} c_{k↑} c_{l↓}  (项3)
                - c†_{i↓} c†_{j↑} c_{k↓} c_{l↑}  (项4)
    """
    # 物理检查：如果 i==j 或 k==l，由于 Pauli 不相容原理，单重态配对直接为 0
    if i == j or k == l:
        zeros_dict = {
            "T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)": 0.0,
            "T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)": 0.0,
            "T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)": 0.0,
            "T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)": 0.0,
            "Total": 0.0
        }
        return 0.0, zeros_dict
        
    # 定义基础单体算符
    cdag_i_up = cdag(hi, i, sz=+1)
    cdag_j_dn = cdag(hi, j, sz=-1)
    cdag_i_dn = cdag(hi, i, sz=-1)
    cdag_j_up = cdag(hi, j, sz=+1)
    
    c_k_up = c(hi, k, sz=+1)
    c_l_dn = c(hi, l, sz=-1)
    c_k_dn = c(hi, k, sz=-1)
    c_l_up = c(hi, l, sz=+1)
    
    # 分别构造 4 个分量算符 (直接将理论符号乘在算符前面)
    term1_op = -1.0 * (cdag_i_up @ cdag_j_dn @ c_k_up @ c_l_dn)
    term2_op =  1.0 * (cdag_i_up @ cdag_j_dn @ c_k_dn @ c_l_up)
    term3_op =  1.0 * (cdag_i_dn @ cdag_j_up @ c_k_up @ c_l_dn)
    term4_op = -1.0 * (cdag_i_dn @ cdag_j_up @ c_k_dn @ c_l_up)
    
    # 辅助函数：求单个算符在守恒空间中的期望值
    def get_expectation(op):
        op_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(op)
        sp_op = op_ed.to_sparse()
        return np.real(np.vdot(psi, sp_op.dot(psi)))
    
    # 计算各项的值
    val1 = get_expectation(term1_op)
    val2 = get_expectation(term2_op)
    val3 = get_expectation(term3_op)
    val4 = get_expectation(term4_op)
    
    # 总值
    total_val = val1 + val2 + val3 + val4
    
    # 将结果装入带标签的字典
    details = {
        "T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)": val1,
        "T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)": val2,
        "T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)": val3,
        "T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)": val4,
        "Total": total_val
    }
    
    return total_val, details

In [3]:
import pandas as pd
import numpy as np

# 假设 psi0, hi, L, 甚至 measure_pairing_correlation 函数都已经如你所定义

# ==========================================
# 批量测量：测量所有相邻 Bond 之间的配对关联 P(ij, kl)
# ==========================================

psi0 = eig_vecs[:, 0]
results = []

print("正在计算全空间 Bond-Bond 配对关联，请稍候...")

# 外层循环：遍历所有可能的参考 Bond (i, j)
for i in range(L - 1):
    j = i + 1
    
    # 内层循环：遍历所有可能的目标 Bond (k, l)
    for k in range(L - 1):
        l = k + 1
        
        # 计算期望值 ⟨Ψ| P_op |Ψ⟩
        val = measure_pairing_correlation(hi, psi0, i, j, k, l)
        
        # 记录数据
        # Distance 定义为起点之间的距离: k - i
        results.append({
            "Ref_Bond": f"({i},{j})",
            "Target_Bond": f"({k},{l})",
            "Ref_i": i,      # 用于构建矩阵的行索引
            "Target_k": k,   # 用于构建矩阵的列索引
            "Distance": k - i,
            "P_val": val
        })

# 1. 转换成 DataFrame (长表格式，适合直接保存为 CSV)
df_pairing = pd.DataFrame(results)

# 如果你需要导出这份完整的数据去画 Distance 的衰减图：
# df_pairing.to_csv("all_pairing_correlation.csv", index=False)

# 2. 将长表转换为 2D 矩阵格式 (非常适合在终端观察全貌)
# 行(index) 为参考键的起点 i，列(columns) 为目标键的起点 k
matrix_df = df_pairing.pivot(index='Ref_i', columns='Target_k', values='P_val')

# 把行和列的标签重命名一下，看起来更清楚
matrix_df.index.name = 'Ref Bond (i, i+1)'
matrix_df.columns.name = 'Target Bond (k, k+1)'

print("\n--- 全空间配对关联矩阵 P(ij, kl) ---")
# 打印矩阵，保留 6 位小数
print(matrix_df.round(6).to_string())

# 如果你想单独看随距离(Distance)的衰减规律，可以根据距离进行分组平均 (针对平移不变的系统)
# 注意：OBC(开边界) 系统的边缘效应很强，分组平均可能抹平细节，一般只挑链中心的 Bond 往外看
print("\n--- 按距离(Distance)汇总的关联值 ---")
dist_df = df_pairing[df_pairing['Distance'] >= 0][['Distance', 'P_val']].groupby('Distance').mean()
print(dist_df.round(6))

正在计算全空间 Bond-Bond 配对关联，请稍候...

--- 全空间配对关联矩阵 P(ij, kl) ---
Target Bond (k, k+1)                                                                                                                                                                                                                                                                                          0                                                                                                                                                                                                                                                                                         1                                                                                                                                                                                                                                                                                        2                                                                           

TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
# 提取基态波函数
psi0 = eig_vecs[:, 0]

# 调用我们刚才修改好的函数，提取出总值和 4 个分项
val, details = measure_pairing_correlation(hi, psi0, 1, 4, 1, 4)

# 打印出极度舒适的排版结果
print("\n==== Python ED: Pairing 测量详情 P(0,2; 0,6) ====")
for formula, v in details.items():
    print(f"{formula:35s} = {v:15.8f}")
print("==================================================\n")


==== Python ED: Pairing 测量详情 P(0,2; 0,6) ====
T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)        =      0.13061639
T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)        =     -0.01707459
T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)        =     -0.01707459
T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)        =      0.13061639
Total                               =      0.22708361



: 

In [ ]:
# 提取基态波函数
psi0 = eig_vecs[:, 0]

# 调用我们刚才修改好的函数，提取出总值和 4 个分项
val, details = measure_pairing_correlation(hi, psi0, 0, 1, 1, 2)

# 打印出极度舒适的排版结果
print("\n==== Python ED: Pairing 测量详情 P(0,1; 1,2) ====")
for formula, v in details.items():
    print(f"{formula:35s} = {v:15.8f}")
print("==================================================\n")


==== Python ED: Pairing 测量详情 P(0,1; 1,2) ====
T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)        =      0.11419482
T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)        =      0.10309890
T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)        =      0.10309890
T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)        =      0.11419482
Total                               =      0.43458745



: 

In [3]:
import pandas as pd
import numpy as np
from netket.experimental.operator import ParticleNumberAndSpinConservingFermioperator2nd
from netket.operator.fermion import number as nc

def measure_local_n_and_sz(hi, psi, i):
    """
    测量第 i 个格点上的平均粒子数 <n_i> 和局域自旋 <Sz_i>
    """
    # 1. 构造上下自旋的粒子数算符
    n_up = nc(hi, i, sz=+1)
    n_dn = nc(hi, i, sz=-1)
    
    # 2. 构造物理观测量算符
    n_op = n_up + n_dn
    Sz_op = 0.5 * (n_up - n_dn)
    
    # 3. 投影到守恒子空间
    n_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(n_op)
    Sz_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(Sz_op)
    
    # 4. 转为稀疏矩阵
    sp_n = n_ed.to_sparse()
    sp_Sz = Sz_ed.to_sparse()
    
    # 5. 计算期望值 ⟨Ψ| O |Ψ⟩
    exp_n = np.vdot(psi, sp_n.dot(psi))
    exp_Sz = np.vdot(psi, sp_Sz.dot(psi))
    
    return np.real(exp_n), np.real(exp_Sz)

# ==========================================
# 批量测量：遍历一维链上的每一个格点
# ==========================================

# 取出基态波函数
psi0 = eig_vecs[:, 0]
results_local = []

# 遍历所有格点 i 从 0 到 L-1
for i in range(L):
    n_val, sz_val = measure_local_n_and_sz(hi, psi0, i)
    results_local.append({
        "Site": i, 
        "<n_i>": n_val, 
        "<S^z_i>": sz_val
    })

# 转换成 DataFrame 方便查看
df_local = pd.DataFrame(results_local)
print("--- 局域粒子数与自旋测量结果 ---")
print(df_local.to_string(index=False))

# ==========================================
# 物理自检 (Sanity Check)
# ==========================================
total_n = df_local["<n_i>"].sum()
total_Sz = df_local["<S^z_i>"].sum()

print("\n--- 物理守恒量自检 ---")
print(f"系统设定电子数: N_up = {N_up}, N_dn = {N_dn}")
print(f"测得总粒子数 Σ<n_i>  = {total_n:.6f}  (理论应为 {N_up + N_dn})")
print(f"测得总自旋 Σ<S^z_i> = {total_Sz:.6f}  (理论应为 {0.5 * (N_up - N_dn)})")

--- 局域粒子数与自旋测量结果 ---
 Site    <n_i>       <S^z_i>
    0 0.812808  1.477984e-15
    1 0.740681 -5.288738e-16
    2 0.680541 -4.753142e-16
    3 0.765969  2.914335e-16
    4 0.765969 -3.816392e-17
    5 0.680541  3.122502e-17
    6 0.740681  1.110223e-16
    7 0.812808 -8.812395e-16

--- 物理守恒量自检 ---
系统设定电子数: N_up = 3, N_dn = 3
测得总粒子数 Σ<n_i>  = 6.000000  (理论应为 6)
测得总自旋 Σ<S^z_i> = -0.000000  (理论应为 0.0)


In [1]:
import numpy as np
import netket as nk
from scipy.sparse.linalg import eigsh
from netket.experimental.operator import ParticleNumberAndSpinConservingFermioperator2nd

# 简化算符名称
from netket.operator.fermion import create as cdag
from netket.operator.fermion import destroy as c

# --- 1. 构造 L=4, N_up=1, N_dn=1 的希尔伯特空间 ---
L = 4
hi = nk.hilbert.SpinOrbitalFermions(L, s=0.5, n_fermions_per_spin=(1, 1))

# --- 2. 构造“玩具哈密顿量”来生成测试态 |Ψ> ---
# H_test = - (对在0,1产生 并 消灭在2,3) - (对在2,3产生 并 消灭在0,1)
H_test = -1.0 * (cdag(hi, 0, sz=+1) @ cdag(hi, 1, sz=-1) @ c(hi, 3, sz=-1) @ c(hi, 2, sz=+1) + 
                 cdag(hi, 2, sz=+1) @ cdag(hi, 3, sz=-1) @ c(hi, 1, sz=-1) @ c(hi, 0, sz=+1))

# 求解这个测试态
H_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(H_test)
sp_H = H_ed.to_sparse()
eig_vals, eig_vecs = eigsh(sp_H, k=1, which="SA")
psi_test = eig_vecs[:, 0]

print(f"测试态能量 (理论应为 -1.0): {eig_vals[0]:.4f}")

# --- 3. 复用我们写的测量函数 ---
def measure_pairing_correlation(hi, psi, i, j, k, l):
    if i == j or k == l:
        return 0.0
    
    Delta_kl = (c(hi, k, sz=+1) @ c(hi, l, sz=-1) - 
                c(hi, k, sz=-1) @ c(hi, l, sz=+1))
    
    Delta_dag_ij = -1.0 * (cdag(hi, i, sz=+1) @ cdag(hi, j, sz=-1) - 
                           cdag(hi, i, sz=-1) @ cdag(hi, j, sz=+1))
    
    P_op = Delta_dag_ij @ Delta_kl
    P_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(P_op)
    
    expectation_value = np.vdot(psi, P_ed.to_sparse().dot(psi))
    return np.real(expectation_value)

# --- 4. 进行终极比对 ---
print("\n--- 关联函数测量结果 ---")

p_01_01 = measure_pairing_correlation(hi, psi_test, 0, 1, 0, 1)
print(f"数值计算 P(01, 01) = {p_01_01:.4f}  | 解析结果 = 0.5000")

p_01_23 = measure_pairing_correlation(hi, psi_test, 0, 1, 2, 3)
print(f"数值计算 P(01, 23) = {p_01_23:.4f}  | 解析结果 = 0.5000")

# 测一个理论上没有粒子的键 (1,2)
p_01_12 = measure_pairing_correlation(hi, psi_test, 0, 1, 1, 2)
print(f"数值计算 P(01, 12) = {p_01_12:.4f}  | 解析结果 = 0.0000")

C:\Users\10783\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: To build H|ψ⟩ use nk.vqs.apply_operator(H, vstate_ψ).

测试态能量 (理论应为 -1.0): -1.0000

--- 关联函数测量结果 ---
数值计算 P(01, 01) = 0.5000  | 解析结果 = 0.5000
数值计算 P(01, 23) = 0.5000  | 解析结果 = 0.5000
数值计算 P(01, 12) = -0.0000  | 解析结果 = 0.0000


In [10]:
import numpy as np
import netket as nk
from math import comb
from scipy.sparse.linalg import eigsh
from netket.experimental.operator import ParticleNumberAndSpinConservingFermioperator2nd

# 为了代码简洁，直接重命名算符，与你的成功案例保持一致
from netket.operator.fermion import create as cdag
from netket.operator.fermion import destroy as c
from netket.operator.fermion import number as nc

# --- 1. 系统参数配置 ---
L = 4                 # 一维链长度
t = 1.0               # 跳跃能级
U = 0.0               # 原位排斥能 (Hubbard U)
n_orbitals = L

# 设置半满配置
if n_orbitals % 2 != 0:
    raise ValueError("n_orbitals 必须为偶数，才能设置 N_up = N_dn = n_orbitals/2")

N_up = n_orbitals // 2
N_dn = n_orbitals // 2

# --- 2. 构造【受限的】费米子希尔伯特空间 ---
# 【核心修复】：使用 n_fermions_per_spin 来分别固定上下自旋的电子数
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals, s=1/2, n_fermions_per_spin=(N_up, N_dn)
)

print('n_orbitals =', n_orbitals)
print('fixed particles (N_up, N_dn) =', (N_up, N_dn))

# --- 3. 构造哈密顿量 ---
H = 0.0  

# 动能项 (Hopping) - OBC 开边界
# sum_{i, sigma} -t * (c^dag_{i, sigma} c_{i+1, sigma} + h.c.)
for i in range(L - 1):
    for sz in (+1, -1):
        H += -t * (cdag(hi, i, sz=sz) @ c(hi, i+1, sz=sz))
        H += -t * (cdag(hi, i+1, sz=sz) @ c(hi, i, sz=sz))

# 相互作用项 (Interaction)
# sum_{i} U * n_{i, up} * n_{i, down}
for i in range(L):
    n_up_i = nc(hi, i, sz=+1)
    n_dn_i = nc(hi, i, sz=-1)
    H += U * (n_up_i @ n_dn_i)

# --- 4. 维度评估与守恒量扇区构建 ---
sector_dim = comb(n_orbitals, N_up) * comb(n_orbitals, N_dn)
max_dim_for_sparse_ed = 2000000  

print("\n--- 系统信息 ---")
print(f"L={L}, U={U}, 电子配置=(N_up={N_up}, N_dn={N_dn})")
print(f"预计该扇区希尔伯特空间维度 (Sector Dim) = {sector_dim}")

if sector_dim > max_dim_for_sparse_ed:
    print("[Skip ED] 该扇区维度过大，不执行 to_sparse()/eigsh 以避免内存溢出。")
else:
    # --- 5. 提取守恒子空间并严格对角化 ---
    # 直接传入 H，底层会自动读取 hi 中的 n_fermions_per_spin 约束
    H_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(H)
    
    print("\n正在构建稀疏矩阵...")
    sp_h = H_ed.to_sparse()
    
    print("正在使用 Lanczos 算法求解基态...")
    # 求解前 k 个最小特征值 (SA: Smallest Algebraic)
    k_vals = min(4, sp_h.shape[0] - 2)
    eig_vals, eig_vecs = eigsh(sp_h, k=k_vals, which="SA")
    
    # 特征值从小到大排序
    eig_vals = np.sort(np.real(eig_vals))

# --- 5. 计算并输出结果 ---
    print("\n--- 计算结果 ---")
    print(f"当前守恒扇区的矩阵实际维度: {sp_h.shape[0]}")
    
    # 基态能量 E0
    e0 = eig_vals[0]
    # 单位格点平均能量
    e_per_site = e0 / L

    print(f"单位格点平均能量 E0/L = {e_per_site:.16f}")
    print(f"最低 {len(eig_vals)} 个能级 = {eig_vals}")

n_orbitals = 4
fixed particles (N_up, N_dn) = (2, 2)

--- 系统信息 ---
L=4, U=0.0, 电子配置=(N_up=2, N_dn=2)
预计该扇区希尔伯特空间维度 (Sector Dim) = 36

正在构建稀疏矩阵...
正在使用 Lanczos 算法求解基态...

--- 计算结果 ---
当前守恒扇区的矩阵实际维度: 36
单位格点平均能量 E0/L = -1.1180339887498953
最低 4 个能级 = [-4.47213595 -3.23606798 -3.23606798 -2.23606798]


In [11]:
import numpy as np

def measure_pairing_correlation(hi, psi, i, j, k, l):
    """
    测量配对关联函数 P(ij, kl) = ⟨Ψ| Δ†_ij Δ_kl |Ψ⟩
    其中 Δ_xy = c_{x↑} c_{y↓} - c_{x↓} c_{y↑}
    
    展开后共计 4 项：
    P(ij, kl) = -(c†_{i↑} c†_{j↓} - c†_{i↓} c†_{j↑}) @ (c_{k↑} c_{l↓} - c_{k↓} c_{l↑})
              = - c†_{i↑} c†_{j↓} c_{k↑} c_{l↓}  (项1)
                + c†_{i↑} c†_{j↓} c_{k↓} c_{l↑}  (项2)
                + c†_{i↓} c†_{j↑} c_{k↑} c_{l↓}  (项3)
                - c†_{i↓} c†_{j↑} c_{k↓} c_{l↑}  (项4)
    """
    # 物理检查：如果 i==j 或 k==l，由于 Pauli 不相容原理，单重态配对直接为 0
    if i == j or k == l:
        zeros_dict = {
            "T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)": 0.0,
            "T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)": 0.0,
            "T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)": 0.0,
            "T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)": 0.0,
            "Total": 0.0
        }
        return 0.0, zeros_dict
        
    # 定义基础单体算符
    cdag_i_up = cdag(hi, i, sz=+1)
    cdag_j_dn = cdag(hi, j, sz=-1)
    cdag_i_dn = cdag(hi, i, sz=-1)
    cdag_j_up = cdag(hi, j, sz=+1)
    
    c_k_up = c(hi, k, sz=+1)
    c_l_dn = c(hi, l, sz=-1)
    c_k_dn = c(hi, k, sz=-1)
    c_l_up = c(hi, l, sz=+1)
    
    # 分别构造 4 个分量算符 (直接将理论符号乘在算符前面)
    term1_op = -1.0 * (cdag_i_up @ cdag_j_dn @ c_k_up @ c_l_dn)
    term2_op =  1.0 * (cdag_i_up @ cdag_j_dn @ c_k_dn @ c_l_up)
    term3_op =  1.0 * (cdag_i_dn @ cdag_j_up @ c_k_up @ c_l_dn)
    term4_op = -1.0 * (cdag_i_dn @ cdag_j_up @ c_k_dn @ c_l_up)
    
    # 辅助函数：求单个算符在守恒空间中的期望值
    def get_expectation(op):
        op_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(op)
        sp_op = op_ed.to_sparse()
        return np.real(np.vdot(psi, sp_op.dot(psi)))
    
    # 计算各项的值
    val1 = get_expectation(term1_op)
    val2 = get_expectation(term2_op)
    val3 = get_expectation(term3_op)
    val4 = get_expectation(term4_op)
    
    # 总值
    total_val = val1 + val2 + val3 + val4
    
    # 将结果装入带标签的字典
    details = {
        "T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)": val1,
        "T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)": val2,
        "T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)": val3,
        "T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)": val4,
        "Total": total_val
    }
    
    return total_val, details

In [27]:
# 提取基态波函数
psi0 = eig_vecs[:, 0]

# 调用我们刚才修改好的函数，提取出总值和 4 个分项
val, details = measure_pairing_correlation(hi, psi0, 0, 2, 2, 3)

# 打印出极度舒适的排版结果
print("\n==== Python ED: Pairing 测量详情 P(0,2; 2,3) ====")
for formula, v in details.items():
    print(f"{formula:35s} = {v:15.8f}")
print("==================================================\n")


==== Python ED: Pairing 测量详情 P(0,2; 2,3) ====
T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)        =      0.00000000
T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)        =     -0.11180340
T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)        =     -0.11180340
T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)        =      0.00000000
Total                               =     -0.22360680



In [48]:
import numpy as np
import netket as nk
from math import comb
from scipy.sparse.linalg import eigsh
from netket.experimental.operator import ParticleNumberAndSpinConservingFermioperator2nd

# 为了代码简洁，直接重命名算符，与你的成功案例保持一致
from netket.operator.fermion import create as cdag
from netket.operator.fermion import destroy as c
from netket.operator.fermion import number as nc

# --- 1. 系统参数配置 ---
L = 8                 # 一维链长度
t = 1.0               # 跳跃能级
U = 8.0               # 原位排斥能 (Hubbard U)
n_orbitals = L

# 设置半满配置
if n_orbitals % 2 != 0:
    raise ValueError("n_orbitals 必须为偶数，才能设置 N_up = N_dn = n_orbitals/2")

N_up = n_orbitals // 2 - 1
N_dn = n_orbitals // 2 - 1

# --- 2. 构造【受限的】费米子希尔伯特空间 ---
# 【核心修复】：使用 n_fermions_per_spin 来分别固定上下自旋的电子数
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals, s=1/2, n_fermions_per_spin=(N_up, N_dn)
)

print('n_orbitals =', n_orbitals)
print('fixed particles (N_up, N_dn) =', (N_up, N_dn))

# --- 3. 构造哈密顿量 ---
H = 0.0  

# 动能项 (Hopping) - OBC 开边界
# sum_{i, sigma} -t * (c^dag_{i, sigma} c_{i+1, sigma} + h.c.)
for i in range(L - 1):
    for sz in (+1, -1):
        H += -t * (cdag(hi, i, sz=sz) @ c(hi, i+1, sz=sz))
        H += -t * (cdag(hi, i+1, sz=sz) @ c(hi, i, sz=sz))

# 相互作用项 (Interaction)
# sum_{i} U * n_{i, up} * n_{i, down}
for i in range(L):
    n_up_i = nc(hi, i, sz=+1)
    n_dn_i = nc(hi, i, sz=-1)
    H += U * (n_up_i @ n_dn_i)

# --- 4. 维度评估与守恒量扇区构建 ---
sector_dim = comb(n_orbitals, N_up) * comb(n_orbitals, N_dn)
max_dim_for_sparse_ed = 2000000  

print("\n--- 系统信息 ---")
print(f"L={L}, U={U}, 电子配置=(N_up={N_up}, N_dn={N_dn})")
print(f"预计该扇区希尔伯特空间维度 (Sector Dim) = {sector_dim}")

if sector_dim > max_dim_for_sparse_ed:
    print("[Skip ED] 该扇区维度过大，不执行 to_sparse()/eigsh 以避免内存溢出。")
else:
    # --- 5. 提取守恒子空间并严格对角化 ---
    # 直接传入 H，底层会自动读取 hi 中的 n_fermions_per_spin 约束
    H_ed = ParticleNumberAndSpinConservingFermioperator2nd.from_fermionoperator2nd(H)
    
    print("\n正在构建稀疏矩阵...")
    sp_h = H_ed.to_sparse()
    
    print("正在使用 Lanczos 算法求解基态...")
    # 求解前 k 个最小特征值 (SA: Smallest Algebraic)
    k_vals = min(4, sp_h.shape[0] - 2)
    eig_vals, eig_vecs = eigsh(sp_h, k=k_vals, which="SA")
    
    # 特征值从小到大排序
    eig_vals = np.sort(np.real(eig_vals))

# --- 5. 计算并输出结果 ---
    print("\n--- 计算结果 ---")
    print(f"当前守恒扇区的矩阵实际维度: {sp_h.shape[0]}")
    
    # 基态能量 E0
    e0 = eig_vals[0]
    # 单位格点平均能量
    e_per_site = e0 / L

    print(f"单位格点平均能量 E0/L = {e_per_site:.16f}")
    print(f"最低 {len(eig_vals)} 个能级 = {eig_vals}")

n_orbitals = 8
fixed particles (N_up, N_dn) = (3, 3)

--- 系统信息 ---
L=8, U=8.0, 电子配置=(N_up=3, N_dn=3)
预计该扇区希尔伯特空间维度 (Sector Dim) = 3136

正在构建稀疏矩阵...
正在使用 Lanczos 算法求解基态...

--- 计算结果 ---
当前守恒扇区的矩阵实际维度: 3136
单位格点平均能量 E0/L = -0.6133250844219010
最低 4 个能级 = [-4.90660068 -4.71796157 -4.4952787  -4.40670521]


In [51]:
# 提取基态波函数
psi0 = eig_vecs[:, 0]

# 调用我们刚才修改好的函数，提取出总值和 4 个分项
val, details = measure_pairing_correlation(hi, psi0, 1, 2, 3, 4)

# 打印出极度舒适的排版结果
print("\n==== Python ED: Pairing 测量详情 P(1,2; 3,4) ====")
for formula, v in details.items():
    print(f"{formula:35s} = {v:15.8f}")
print("==================================================\n")


==== Python ED: Pairing 测量详情 P(1,2; 3,4) ====
T1 (- c†_i↑ c†_j↓ c_k↑ c_l↓)        =      0.00291038
T2 (+ c†_i↑ c†_j↓ c_k↓ c_l↑)        =     -0.00766865
T3 (+ c†_i↓ c†_j↑ c_k↑ c_l↓)        =     -0.00766865
T4 (- c†_i↓ c†_j↑ c_k↓ c_l↑)        =      0.00291038
Total                               =     -0.00951655

